The purpose of this template file is to do pre-processing and feature selection.

In [1]:
!pip install braindecode[hub]==1.8.1
import braindecode
!pip install open-clip-torch
!pip install PyWavelets -q

/usr/local/lib/python3.13/dist-packages/braindecode/models/eegpt.py:497: FutureWarning: Montage name 'standard_1020' is deprecated and will be removed in MNE 1.14. Use 'colin27_1020' instead.
  montage = mne.channels.make_standard_montage("standard_1020")
/usr/local/lib/python3.13/dist-packages/braindecode/models/eegpt.py:1452: FutureWarning: Montage name 'standard_1020' is deprecated and will be removed in MNE 1.14. Use 'colin27_1020' instead.
  montage = make_standard_montage("standard_1020")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.1 MB/s eta 0:00:00


In [2]:
import os
import importlib
import sys
from pathlib import Path
import json
from typing import Callable

from tqdm import tqdm
import numpy as np
import h5py
import cv2

from scipy import signal
from scipy.signal import decimate

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

import mne
from mne.io import Raw

In [3]:
import labram
import biot
import cbramod
import classical_features
import labram_special
import luna
import openclip
import dino
import videomae
from preprocessing import bandpass_filter, notch_filter, smoothing, ica

import helpers
from helpers import save_features, load_features

importlib.reload(helpers)
importlib.reload(classical_features)

<module 'classical_features' from '/content/classical_features.py'>

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
prefix     = Path("/content/drive/MyDrive/schubki2026_data")
EMD        = prefix / Path('EMD')
STIM_TRN_DIR   = prefix / Path('stimuli_train')
STIM_TST_DIR   = prefix / Path('stimuli_test')
FEAT_DIR   = prefix / Path('features')
FRAME_DIR  = prefix / Path('frames')
TRAIN_DIR  = prefix / Path('trn')
TEST_DIR  = prefix / Path('tst')
SFREQ      = 500                        # Hz, from metaDat['times']
NUM_VIDS   = 1102
CH_NAMES   = ["Pz", "P3", "P7", "O1", "Oz", "O2", "P4", "P8", "P1", "P5", "PO7", "PO3", "POz", "PO4", "PO8", "P6", "P2"]

low_filter = 0.1
high_filter = 75.0
notch_freqs= 50.0
do_smoothing = False
smooth_lp = 35.0

# Loading data

In [7]:
def load_subject(sub):
    trn = np.load(f"{TRAIN_DIR}/sub-{sub:02d}_trn.npz")
    stim_ids = np.array(sorted(trn.files, key=int), dtype=int)
    data = np.stack([trn[str(i)] for i in stim_ids])  # (n_trials, 17, 1500)
    return data, stim_ids

def load_subject_test(sub):
    tst_eeg = np.load(f'{TEST_DIR}/sub-{sub:02d}_tst.npy')
    return tst_eeg


def bandpass(x, sfreq, l_freq, h_freq, order=4):
    sos = signal.butter(order, [l_freq, h_freq], btype="bandpass", fs=sfreq, output="sos")
    return signal.sosfiltfilt(sos, x, axis=-1) # zero-phase


def notch(x, sfreq, freq=50.0, q=30.0):
    b, a = signal.iirnotch(freq, q, fs=sfreq)
    return signal.filtfilt(b, a, x, axis=-1)


def preprocess_baseline(x, sfreq):
    DEC = 5
    x = decimate(x, DEC, axis=-1, ftype='fir', zero_phase=True)
    return x

def preprocess(x, sfreq):
    x = bandpass(x, sfreq, 0.5, 75.0)
    x = notch(x, sfreq, 50.0)
    x = x - x.mean(axis=1, keepdims=True)  # average reference (over channels)
    return x

X_list, trial_size, stim_list = [], [], []
for sub in tqdm(range(1,6+1)):
    data, stim_ids = load_subject(sub)
    x = preprocess(data, SFREQ)
    X_list.append(x)
    trial_size.append(data.shape[0])
    stim_list.append(stim_ids)
X = np.concatenate(X_list)
stim_ids = np.concatenate(stim_list)

X_test_list, trial_size_test = [], []
for sub in tqdm(range(1,6+1)):
    data = load_subject_test(sub)
    x = preprocess(data, SFREQ)
    trial_size_test.append(data.shape[0])
    X_test_list.append(x)
X_test = np.concatenate(X_test_list)

100%|██████████| 6/6 [00:09<00:00,  1.63s/it]


# Feature engineering

In [8]:
importlib.reload(classical_features)

# Band power
BANDS = {
    "delta": (1.0, 4.0),
    "theta": (4.0, 8.0),
    "alpha_low": (8.0, 10.0),
    "alpha_mid": (10.0, 12.0),
    "alpha_high": (12.0, 13.0),
    "beta_low": (13.0, 21.0),
    "beta_high": (21.0, 30.0),
    "gamma_low": (30.0, 37.5),
    "gamma_high": (37.5, 45.0),
}
band_power = classical_features.band_power(X, SFREQ, bands=BANDS)
band_power = band_power.reshape(len(X), -1)

band_power_test = classical_features.band_power(X_test, SFREQ, bands=BANDS)
band_power_test = band_power_test.reshape(len(X_test), -1)

# Morlet
morlet, _, _ = classical_features.cwt_morlet(X, SFREQ, fmin=3.0, fmax=40.0, freq_step=10.0, decim=10)
morlet = morlet.reshape(len(X), -1)

scaler_morlet = StandardScaler(copy=False)
scaler_morlet.fit(morlet)
morlet = scaler_morlet.transform(morlet)

pca_morlet = PCA(n_components=300, svd_solver="randomized", copy=False, random_state=1337)
pca_morlet.fit(morlet)
morlet_pca = pca_morlet.transform(morlet)

morlet_test, _, _ = classical_features.cwt_morlet(X_test, SFREQ, fmin=3.0, fmax=40.0, freq_step=10.0, decim=10)
morlet_test = morlet_test.reshape(len(X_test), -1)
morlet_test = scaler_morlet.transform(morlet_test)
morlet_pca_test = pca_morlet.transform(morlet_test)

# DWT
dwt = classical_features.dwt(X)
dwt = dwt.reshape(len(X), -1)

dwt_test = classical_features.dwt(X_test)
dwt_test = dwt_test.reshape(len(X_test), -1)

# HFD
hfd = classical_features.hfd(X, kmax=5)
hfd_test = classical_features.hfd(X_test, kmax=5)

scaler_hfd = StandardScaler()
hfd_scaled = scaler_hfd.fit_transform(hfd)
hfd_test_scaled = scaler_hfd.transform(hfd_test)

In [9]:
print("train")
print(band_power.shape)
print(morlet_pca.shape)
print(dwt.shape)
print(hfd_scaled.shape)

print("test")
print(band_power_test.shape)
print(morlet_pca_test.shape)
print(dwt_test.shape)
print(hfd_test_scaled.shape)

train
(5650, 153)
(5650, 300)
(5650, 102)
(5650, 17)
test
(612, 153)
(612, 300)
(612, 102)
(612, 17)


In [31]:
def separate_features(path, feature, trial_size):
    path.mkdir(parents=True, exist_ok=True)
    prev = 0
    for i, sz in tqdm(enumerate(trial_size), leave=True):
        p = path / f"sub-{i+1}.h5"
        if not p.exists():
            save_features(
                p,
                feature[prev:prev+sz],
                stim_ids[prev:prev+sz]
            )
        prev += sz

def save_cfeature(path, cfeature, stim_ids, trial_size):
    path.mkdir(parents=True, exist_ok=True)
    cfeature = np.concatenate(
        [band_power, morlet_pca, dwt],
        axis=1,
    )
    if not path.exists():
        save_features(path / "features.h5", cfeature, stim_ids)
    separate_features(path, cfeature, trial_size)

save_cfeature(
    Path("/content/drive/MyDrive/eeg_features/schubki_classical_feature_bp_morlet_dwt"),
    np.concatenate(
        [band_power, morlet_pca, dwt],
        axis=1,
    ),
    stim_ids,
    trial_size
)

save_cfeature(
    Path("/content/drive/MyDrive/eeg_features/schubki_classical_feature_bp_morlet_dwt_test"),
    np.concatenate(
        [band_power_test, morlet_pca_test, dwt_test],
        axis=1,
    ),
    np.zeros(1),
    trial_size_test
)

# version with hfd
save_cfeature(
    Path("/content/drive/MyDrive/eeg_features/schubki_classical_feature_bp_morlet_dwt_hfd"),
    np.concatenate(
        [band_power, morlet_pca, dwt, hfd_scaled],
        axis=1,
    ),
    stim_ids,
    trial_size
)

save_cfeature(
    Path("/content/drive/MyDrive/eeg_features/schubki_classical_feature_bp_morlet_dwt_hfd_test"),
    np.concatenate(
        [band_power_test, morlet_pca_test, dwt_test, hfd_test_scaled],
        axis=1,
    ),
    np.zeros(1),
    trial_size_test
)

6it [00:00, 1380.77it/s]
6it [00:00, 1130.18it/s]
6it [00:00, 1128.21it/s]
6it [00:00, 1174.00it/s]


# EEG Embedding via foundation models

In [23]:
# All input to embeddings is given in format (n_trials, n_channels, n_times)
# with unit micro Volt
"""
(
    Path("/content/drive/MyDrive/eeg_features/schubki_biot.h5"),
    lambda X, ch_names: biot.biot_embed(X, 500, ch_names, batch_size = 32, device = "cuda"),
),
"""
embeddings = [
    (
        Path("/content/drive/MyDrive/eeg_features/schubki_luna/"),
        lambda X, ch_names: luna.luna_embed(X, 500, ch_names, batch_size = 32, device = "cuda"),
    ),
    (
        Path("/content/drive/MyDrive/eeg_features/schubki_labram/"),
        lambda X, ch_names: labram.labram_embed(X, 500, ch_names, batch_size = 32, device = "cuda"),
    ),
    (
        Path("/content/drive/MyDrive/eeg_features/schubki_cbramod/"),
        lambda X, _: cbramod.cbramod_embed(X, 500, batch_size = 32, device = "cuda"),
    ),
    (
        Path("/content/drive/MyDrive/eeg_features/schubki_labram_special/"),
        lambda X, ch_names: labram_special.labram_embed(X, 500, ch_names, batch_size = 32, device = "cuda"),
    ),
]

embeddings_test = [
    (
        Path("/content/drive/MyDrive/eeg_features/schubki_luna_test/"),
        lambda X, ch_names: luna.luna_embed(X, 500, ch_names, batch_size = 32, device = "cuda"),
    ),
    (
        Path("/content/drive/MyDrive/eeg_features/schubki_labram_test/"),
        lambda X, ch_names: labram.labram_embed(X, 500, ch_names, batch_size = 32, device = "cuda"),
    ),
    (
        Path("/content/drive/MyDrive/eeg_features/schubki_cbramod_test/"),
        lambda X, _: cbramod.cbramod_embed(X, 500, batch_size = 32, device = "cuda"),
    ),
    (
        Path("/content/drive/MyDrive/eeg_features/schubki_labram_special_test/"),
        lambda X, ch_names: labram_special.labram_embed(X, 500, ch_names, batch_size = 32, device = "cuda"),
    ),
]

In [26]:
for path, embed in tqdm(embeddings):
    print(f"Start {path}")
    path.mkdir(parents=True, exist_ok=True)
    res = embed(X, CH_NAMES)
    prev = 0
    for i, sz in tqdm(enumerate(trial_size), leave=True):
        save_features(
            path / f"sub-{i+1}.h5",
            res[prev:prev+sz],
            stim_ids[prev:prev+sz],
            exists_ok=True
        )
        prev += sz
    save_features(path / "features.h5", res, stim_ids, exists_ok=True)

# for test data
for path, embed in tqdm(embeddings_test):
    print(f"Start {path}")
    path.mkdir(parents=True, exist_ok=True)
    res = embed(X_test, CH_NAMES)
    prev = 0
    for i, sz in tqdm(enumerate(trial_size), leave=True):
        save_features(
            path / f"sub-{i+1}.h5",
            res[prev:prev+sz],
            stim_ids[prev:prev+sz],
            exists_ok=True
        )
        prev += sz
    save_features(path / "features.h5", res, stim_ids, exists_ok=True)

  0%|          | 0/4 [00:00<?, ?it/s]

Start /content/drive/MyDrive/eeg_features/schubki_luna
Kept 17/17 channels.



6it [00:00, 71.79it/s]
 25%|██▌       | 1/4 [00:24<01:13, 24.65s/it]

Start /content/drive/MyDrive/eeg_features/schubki_labram
Kept 0/17 channels.



6it [00:00, 62.11it/s]
 50%|█████     | 2/4 [00:52<00:53, 26.74s/it]

...


6it [00:00, 99.71it/s]
 50%|█████     | 2/4 [00:04<00:04,  2.50s/it]

Start /content/drive/MyDrive/eeg_features/schubki_cbramod_test



6it [00:00, 115.49it/s]
 75%|███████▌  | 3/4 [00:06<00:02,  2.00s/it]

Start /content/drive/MyDrive/eeg_features/schubki_labram_special_test



6it [00:00, 85.37it/s]
100%|██████████| 4/4 [00:07<00:00,  1.94s/it]


# Video embedding

In [ ]:
def get_video_by_id(video_id: int):
    if video_id <= 1000:
        video_dir = STIM_TRN_DIR
    else:
        video_dir = STIM_TST_DIR
    file = next(Path(video_dir).rglob(f"{video_id:04d}*"), None)

    cap = cv2.VideoCapture(str(file))
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    ret = np.empty((frame_count, height, width, 3), np.dtype("uint8"))

    # CAP_PROP_FRAME_COUNT is only an estimate; keep only the frames actually read.
    n_read = 0
    for i in range(frame_count):
        has_next, frame = cap.read()
        if not has_next:
            break
        ret[i] = frame
        n_read += 1
    cap.release()
    return ret[:n_read]

def compute_video_features(
    feature_dir: Path,
    embed: Callable[[np.ndarray, list[str]], np.ndarray]
):
    batch_size = 16
    for video_id in tqdm(range(1, NUM_VIDS+1)):
        feature_path = feature_dir / f"feature-{video_id}.h5"
        if feature_path.exists():
            continue
        res = embed([get_video_by_id(video_id)])
        res = res.squeeze(0)
        save_features(feature_path, res, np.zeros(1))

# All input to embeddings is given in format (n_trials, n_channels, n_times)
# with unit micro Volt
video_embeddings = [
    (
        Path("/content/drive/MyDrive/eeg_features/schubki_video_openclip"),
        lambda X: openclip.openclip_embed(X, device="cuda")
    ),
    (
        Path("/content/drive/MyDrive/eeg_features/schubki_video_dino"),
        lambda X: dino.dinov2_embed(X, device="cuda")
    ),
    (
        Path("/content/drive/MyDrive/eeg_features/schubki_video_videomae"),
        lambda X: videomae.videomae_embed(X, device="cuda")
    ),
]

for dir, func in video_embeddings:
    print(f"Start {dir}")
    dir.mkdir(exist_ok=True, parents=True)
    compute_video_features(dir, func)

# Data augmentation (if needed later)








In [ ]:
from braindecode.augmentation import (
    FTSurrogate,
    FrequencyShift,
    ChannelsDropout,
    TimeReverse,
    Compose,
)
import torch

transforms = [
    FTSurrogate(probability=0.5, phase_noise_magnitude=0.8, channel_indep=False),
    FrequencyShift(probability=0.5, sfreq=SFREQ, max_delta_freq=1.5),
    ChannelsDropout(probability=0.5, p_drop=0.2),
    TimeReverse(probability=0.5),
]

composed_transforms = Compose(transforms)

def augment_eeg_batch(X: np.ndarray, Y: np.ndarray, n_copies: int = 1):
    """Generate original + n_copies augmented versions of the EEG trials.

    X: (n_trials, n_channels, n_times) in uV, numpy
    Y: (n_trials,) labels
    Returns concatenated X_aug, Y_aug with shape roughly (n_trials * (1+n_copies), ...)
    """
    if n_copies <= 0:
        return X, Y

    X_t = torch.from_numpy(X).float()
    Y_t = torch.from_numpy(Y).long()

    Xs = [X]
    Ys = [Y]

    for _ in range(n_copies):
        X_aug, Y_aug = composed_transforms(X_t, Y_t)
        Xs.append(X_aug.cpu().numpy())
        Ys.append(Y_aug.cpu().numpy())

    return np.concatenate(Xs, axis=0), np.concatenate(Ys, axis=0)

print("Augmentation transforms ready. Use augment_eeg_batch before embedding.")
